In [1]:
!pip install -q qdrant-client transformers peft pillow openai
!pip uninstall -y torchao

In [2]:
"""
VLM-as-a-Judge 검색 평가 (LookMatch) — 등급 + 유의성 + 유사도 margin + 오차감소
==============================================================================
실제 사용자 검색어로 검색하고, GPT-4o가 각 결과의 관련도를 0~3 등급 판정.
→ Recall@K, nDCG@K (등급 반영)
→ 원본 vs 파인튜닝 쿼리별 nDCG를 모아 Wilcoxon paired 검정 (유의성)
→ [추가] 유사도 분리도(margin): 관련(등급≥2) 평균 유사도 − 무관(등급≤1) 평균 유사도
         · 절대 유사도값이 아니라 '관련/무관을 얼마나 벌리는가'로 정당하게 비교
→ [추가] 상대 오차감소: near-ceiling에서 (1-nDCG) 기준으로 개선폭을 공정하게 표현

등급 루브릭 (색상·종류·핏):
  3 = 색상·종류·핏 모두 일치 (완벽)
  2 = 2개 일치
  1 = 1개 일치 (약간 관련)
  0 = 무관

모델 3개 비교: 원본 / Qwen 파인튜닝 / 네이버 파인튜닝

설치:
  pip install qdrant-client transformers peft pillow pandas tqdm openai numpy scipy
실행:
  1. 각 모델로 검증셋 인덱싱 (search_lookmatch.py, VAL_ONLY=True)
  2. python evaluate_vlm_judge.py

주의: OPENAI_API_KEY는 환경변수로만 관리 (코드 하드코딩 금지)
  PowerShell:  $env:OPENAI_API_KEY = "sk-..."
  bash:        export OPENAI_API_KEY="sk-..."
"""

import os
import math
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm


# ============================================================
# 설정
# ============================================================
try:
    import google.colab  # noqa
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DATA_DIR = Path("/content/drive/MyDrive/clothes_dataset")
except ImportError:
    DATA_DIR = Path(r"C:\Users\Hyunuk\Desktop\lookmatch_project\clothes_dataset")
CROPS_SUBDIR = "crops"
LOCAL_CROPS = DATA_DIR / "crops"
QDRANT_PATH = str(DATA_DIR / "qdrant_db")

SIGLIP_MODEL = "google/siglip2-so400m-patch16-512"
JUDGE_MODEL = "gpt-4o"   # OpenAI API (GPU 불필요)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")   # ← 환경변수로만
OPENAI_API_KEY = ""

MODELS = [
    ("원본",        "clothes_original_valonly", None),
    ("Qwen파인튜닝", "clothes_valonly",          str(DATA_DIR / "siglip2_so400m_finetuned_full_qwen")),
    ("네이버파인튜닝", "clothes_naver_valonly",    str(DATA_DIR / "siglip2_so400m_finetuned_full_naver")),
]
BASELINE_NAME = "원본"   # 유의성/오차감소 기준 모델

# 실제 쇼핑 검색어 (60개: 색상·핏·소재·복합·hard 단계별)
QUERIES = [
    # 1. 기본 색상 + 카테고리
    "베이지 슬랙스", "차콜 니트", "아이보리 티셔츠", "브라운 셔츠", "핑크 원피스",
    "카키 카고 팬츠", "네이비 코트", "크림 가디건", "블랙 조거 팬츠", "블루 데님 자켓",
    # 2. 색상 + 핏 + 카테고리
    "베이지 와이드 슬랙스", "차콜 세미오버핏 니트", "아이보리 오버핏 반팔 티셔츠",
    "브라운 오버핏 셔츠", "멜란지 그레이 오버핏 후드티", "네이비 오버핏 피케 티셔츠",
    "그레이 세미크롭 니트", "베이지 오버핏 린넨 셔츠", "블랙 와이드 조거 팬츠",
    "워싱 블루 크롭 데님 자켓",
    # 3. 디자인 / 디테일 속성
    "베이지 투턱 슬랙스", "중청 부츠컷 데님", "진청 하이웨스트 데님", "블랙 A라인 미니 스커트",
    "아이보리 플로럴 블라우스", "핑크 플리츠 원피스", "오프화이트 퀼팅 패딩",
    "그레이 케이블 니트", "브라운 체크 셔츠", "아이보리 퍼프 블라우스",
    # 4. 소재 / 구조 속성
    "베이지 린넨 셔츠", "네이비 울 코트", "크림 골지 니트", "블랙 밴딩 팬츠",
    "브라운 린넨 셔츠", "차콜 케이블 니트", "네이비 니트 가디건", "오프화이트 퀼팅 자켓",
    "그레이 골지 가디건", "블랙 데님 스커트",
    # 5. 복합 속성
    "베이지 투턱 와이드 슬랙스", "중청 하이웨스트 부츠컷 청바지", "아이보리 플로럴 퍼프 블라우스",
    "블랙 하이웨스트 A라인 미니 스커트", "핑크 플리츠 롱 원피스", "차콜 하이넥 골지 니트",
    "카키 세미오버핏 셔츠", "오프화이트 롱 퀼팅 패딩", "네이비 울 롱 코트",
    "카키 와이드 카고 팬츠",
    # 6. Hard / 세부 속성 구분
    "진청 하이웨스트 부츠컷 데님 팬츠", "차콜 투턱 와이드 슬랙스", "네이비 스트라이프 오버핏 셔츠",
    "네이비 오버핏 피케 카라 티셔츠", "그레이 세미크롭 케이블 니트", "블랙 하이웨스트 와이드 슬랙스",
    "브라운 체크 오버핏 셔츠", "아이보리 플로럴 퍼프 반팔 블라우스", "블랙 하이웨스트 미니 스커트",
    "올리브 와이드 카고 팬츠",
]

TOP_K = 20
K_LIST = [1, 5, 10, 20]
MAX_REL = 3          # 최고 등급 (0~3)
REL_THRESHOLD = 2    # 등급 이 값 이상 = '관련' (margin·recall 기준)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def resolve_image_path(crop_path):
    fname = str(crop_path).replace("\\", "/").split("/")[-1]
    for cand in [LOCAL_CROPS / fname, Path(crop_path), DATA_DIR / CROPS_SUBDIR / fname]:
        if Path(cand).exists():
            return str(cand)
    return None


# ============================================================
# SigLIP2 (검색용 임베딩)
# ============================================================
def get_siglip(finetuned_dir):
    """캐시 없이 매번 로드 → 평가 후 GPU 비우기 (OOM 방지)."""
    from transformers import AutoModel, AutoProcessor
    model = AutoModel.from_pretrained(SIGLIP_MODEL).to(DEVICE)
    if finetuned_dir and Path(finetuned_dir).exists():
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, finetuned_dir).to(DEVICE)
    model.eval()
    proc = AutoProcessor.from_pretrained(SIGLIP_MODEL)
    return model, proc


def sig_base(model):
    if hasattr(model, "base_model") and hasattr(model.base_model, "model"):
        return model.base_model.model
    return model


def _to_tensor(out):
    if torch.is_tensor(out):
        return out
    for attr in ["text_embeds", "image_embeds", "pooler_output"]:
        if hasattr(out, attr):
            v = getattr(out, attr)
            if torch.is_tensor(v):
                return v
    raise TypeError(f"임베딩 못 찾음: {type(out)}")


@torch.no_grad()
def embed_text(model, proc, text):
    import torch.nn.functional as F
    base = sig_base(model)
    inp = proc(text=[text], return_tensors="pt", padding="max_length",
               max_length=64, truncation=True).to(DEVICE)
    emb = _to_tensor(base.get_text_features(**inp))
    return F.normalize(emb.float(), dim=-1).cpu().numpy()[0]


# ============================================================
# GPT-4o 등급 판정 (VLM-as-Judge, graded)
# ============================================================
_CLIENT = {}
def get_client():
    if "c" not in _CLIENT:
        import openai
        key = os.environ.get("OPENAI_API_KEY", OPENAI_API_KEY)
        if not key:
            raise RuntimeError("OPENAI_API_KEY 환경변수가 없습니다. $env:OPENAI_API_KEY 로 설정하세요.")
        _CLIENT["c"] = openai.OpenAI(api_key=key)
        print(f"판정: {JUDGE_MODEL} (OpenAI API)")
    return _CLIENT["c"]


def _img_to_b64(image_path):
    import base64
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode()


@torch.no_grad()
def judge_relevance(query, image_path):
    """GPT-4o가 관련도를 0~3 등급으로 판정. 색상·종류·핏 각각 확인."""
    client = get_client()
    img_b64 = _img_to_b64(image_path)
    prompt = (
        f'Rate how well this clothing image matches the search query "{query}".\n'
        f'Check three aspects separately: color, garment type, and fit/silhouette.\n'
        f'Scoring:\n'
        f'  3 = all three (color, type, fit) match\n'
        f'  2 = exactly two match\n'
        f'  1 = only one matches\n'
        f'  0 = none match (irrelevant)\n'
        f'Answer with a single number 0, 1, 2, or 3. No other text.'
    )
    try:
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "image_url",
                     "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}},
                    {"type": "text", "text": prompt},
                ],
            }],
            max_tokens=8, temperature=0,
        )
        text = resp.choices[0].message.content
    except Exception as e:
        print(f"[API 오류] {e}")
        return 0
    for ch in str(text):
        if ch in "0123":
            return int(ch)
    return 0


# ============================================================
# 한 모델 평가 (graded)
#   반환: metrics(평균+margin), perq(쿼리별 nDCG 리스트)
# ============================================================
def eval_model(name, collection, finetuned_dir):
    from qdrant_client import QdrantClient

    print(f"\n{'='*60}")
    print(f"[{name}] 평가 (컬렉션: {collection})")
    print(f"{'='*60}")

    model, proc = get_siglip(finetuned_dir)
    client = QdrantClient(path=QDRANT_PATH)

    all_recall = {k: [] for k in K_LIST}
    all_ndcg = {k: [] for k in K_LIST}   # 쿼리별 nDCG (유의성 검정용)
    sims_pos, sims_neg = [], []          # 유사도 margin용 (관련/무관)

    for query in tqdm(QUERIES, desc=f"{name}"):
        # 1. 검색 (r.score = 코사인 유사도)
        q_emb = embed_text(model, proc, query)
        results = client.query_points(
            collection_name=collection, query=q_emb.tolist(), limit=TOP_K).points

        # 2. GPT-4o 등급 판정 (0~3)
        rels, scores = [], []
        for r in results:
            img_path = resolve_image_path(r.payload.get("crop_path", ""))
            rel = judge_relevance(query, img_path) if img_path else 0
            rels.append(rel)
            scores.append(float(r.score))

        # 2-b. 유사도 분리도(margin)용 수집: 관련(등급≥2) vs 무관(등급≤1)
        for rel, sc in zip(rels, scores):
            if rel >= REL_THRESHOLD:
                sims_pos.append(sc)
            else:
                sims_neg.append(sc)

        # 3. Recall@K + nDCG@K (등급 반영)
        for k in K_LIST:
            top_rels = rels[:k]
            hit = 1 if any(rel >= REL_THRESHOLD for rel in top_rels) else 0
            all_recall[k].append(hit)

            dcg = sum((2 ** rel - 1) / math.log2(i + 2)
                      for i, rel in enumerate(top_rels))
            ideal = sorted(top_rels, reverse=True)   # 검색된 top_k 기준 이상 정렬
            idcg = sum((2 ** rel - 1) / math.log2(i + 2)
                       for i, rel in enumerate(ideal))
            ndcg = dcg / idcg if idcg > 0 else 0
            all_ndcg[k].append(ndcg)

    # 유사도 margin 집계
    pos_mean = float(np.mean(sims_pos)) if sims_pos else float("nan")
    neg_mean = float(np.mean(sims_neg)) if sims_neg else float("nan")
    margin = pos_mean - neg_mean if (sims_pos and sims_neg) else float("nan")

    metrics = {
        "Recall": {k: sum(v) / len(v) for k, v in all_recall.items()},
        "nDCG": {k: sum(v) / len(v) for k, v in all_ndcg.items()},
        "sim": {"pos": pos_mean, "neg": neg_mean, "margin": margin,
                "n_pos": len(sims_pos), "n_neg": len(sims_neg)},
    }
    perq = {k: list(all_ndcg[k]) for k in K_LIST}

    import gc
    del model, proc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    return metrics, perq


# ============================================================
# 유의성 검정 + 오차감소 (원본 vs 각 파인튜닝)
# ============================================================
def compare_significance(perq_all):
    """원본 대비 각 파인튜닝의 쿼리별 nDCG를 Wilcoxon signed-rank로 검정.
    - 평균차, 승-무-패(동점 포함), p값, 유효표본
    - [추가] 상대 오차감소 = (오차_원본 − 오차_파인튜닝) / 오차_원본,  오차 = 1 − nDCG
      · near-ceiling(0.9x)에서 '+0.01'보다 공정한 개선폭 표현
    """
    try:
        from scipy.stats import wilcoxon
    except ImportError:
        print("\n[유의성 검정 생략] scipy 미설치 → pip install scipy")
        return

    if BASELINE_NAME not in perq_all:
        print(f"\n[유의성 검정 생략] 기준 모델 '{BASELINE_NAME}' 결과 없음")
        return

    print(f"\n{'='*60}")
    print(f"유의성 검정 + 오차감소 — 각 모델 vs [{BASELINE_NAME}]")
    print(f"검색어 {len(QUERIES)}개 · Wilcoxon signed-rank (쿼리별 nDCG, paired)")
    print(f"{'='*60}")

    base = perq_all[BASELINE_NAME]
    for name, perq in perq_all.items():
        if name == BASELINE_NAME:
            continue
        print(f"\n[{name}] vs [{BASELINE_NAME}]")
        for k in K_LIST:
            a = np.array(perq[k], dtype=float)    # 파인튜닝
            b = np.array(base[k], dtype=float)    # 원본
            d = a - b
            wins = int((d > 1e-12).sum())
            losses = int((d < -1e-12).sum())
            ties = int((np.abs(d) <= 1e-12).sum())
            mean_d = float(d.mean())
            n_eff = wins + losses

            # 상대 오차감소 (평균 nDCG 기준)
            err_base = 1.0 - float(b.mean())
            err_ft = 1.0 - float(a.mean())
            if err_base > 1e-9:
                red = (err_base - err_ft) / err_base * 100.0
                red_str = f"오차감소 {red:+.0f}%"
            else:
                red_str = "오차감소 N/A(천장)"

            if n_eff == 0:
                pstr, verdict = "  N/A", "모든 쿼리 동점 → 검정 불가"
            else:
                try:
                    _, p = wilcoxon(a, b)
                    sig = "유의 O" if p < 0.05 else "유의 X"
                    pstr, verdict = f"{p:.3f}", f"{sig} · 유효표본 {n_eff}개"
                except Exception as e:
                    pstr, verdict = "  err", f"검정 오류: {e}"

            print(f"  nDCG@{k:<2d}: 평균차={mean_d:+.4f} | 승-무-패={wins}-{ties}-{losses} "
                  f"| p={pstr} | {red_str} → {verdict}")

    print(f"\n※ Wilcoxon은 승패 '수'가 아니라 차이의 '크기(부호순위)'로 판정 "
          f"→ 승이 많아도 폭이 상쇄되면 유의 X.")
    print(f"※ '유의 X'는 '효과 없음'이 아니라 '이 표본으로는 증명 불가'.")


def main():
    print(f"디바이스: {DEVICE}")
    print(f"검색어 {len(QUERIES)}개, 모델 {len(MODELS)}개")
    print(f"판정: 0~3 등급 (색상·종류·핏 루브릭)")

    results = {}
    perq_all = {}
    for name, coll, fdir in MODELS:
        try:
            metrics, perq = eval_model(name, coll, fdir)
            results[name] = metrics
            perq_all[name] = perq
        except Exception as e:
            print(f"[{name}] 실패: {e}")

    print(f"\n{'='*60}")
    print("VLM-as-Judge 최종 비교 (등급 판정, 실제 검색어)")
    print(f"{'='*60}")
    for name, m in results.items():
        r = " / ".join(f"R@{k}={v:.3f}" for k, v in m["Recall"].items())
        n = " / ".join(f"nDCG@{k}={v:.3f}" for k, v in m["nDCG"].items())
        s = m["sim"]
        print(f"  {name:14s}")
        print(f"    Recall(등급≥2): {r}")
        print(f"    nDCG(등급형)   : {n}")
        # 유사도 분리도: 관련/무관 평균과 그 차이(margin). 클수록 관련·무관을 잘 벌림
        print(f"    유사도 margin  : 관련={s['pos']:.3f} − 무관={s['neg']:.3f} "
              f"= {s['margin']:+.3f}  (관련 {s['n_pos']}장 / 무관 {s['n_neg']}장)")

    # 유의성 + 오차감소
    compare_significance(perq_all)


if __name__ == "__main__":
    main()

디바이스: cuda
검색어 60개, 모델 3개
판정: 0~3 등급 (색상·종류·핏 루브릭)

[원본] 평가 (컬렉션: clothes_original_valonly)


Loading weights: 100%|██████████| 888/888 [00:00<00:00, 5833.38it/s]
C:\Users\Hyunuk\AppData\Local\Temp\ipykernel_24060\1653424010.py:225: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <clothes> contains 71637 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path=QDRANT_PATH)
C:\Users\Hyunuk\AppData\Local\Temp\ipykernel_24060\1653424010.py:225: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <clothes_original> contains 71637 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path=QDRANT_PATH)
원본:   0%|          | 0/60 [00:00<?, ?it/s]

판정: gpt-4o (OpenAI API)


원본: 100%|██████████| 60/60 [19:07<00:00, 19.12s/it]



[Qwen파인튜닝] 평가 (컬렉션: clothes_valonly)


Loading weights: 100%|██████████| 888/888 [00:00<00:00, 3224.67it/s]
Exception in thread Thread-6 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Hyunuk\anaconda3\envs\myproject\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\Hyunuk\anaconda3\envs\myproject\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Hyunuk\anaconda3\envs\myproject\Lib\subprocess.py", line 1597, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte
C:\Users\Hyunuk\AppData\Local\Temp\ipykernel_24060\1653424010.py:225: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <clothes> contains 71637 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantCl


[네이버파인튜닝] 평가 (컬렉션: clothes_naver_valonly)


네이버파인튜닝: 100%|██████████| 60/60 [18:41<00:00, 18.69s/it]



VLM-as-Judge 최종 비교 (등급 판정, 실제 검색어)
  원본            
    Recall(등급≥2): R@1=0.967 / R@5=1.000 / R@10=1.000 / R@20=1.000
    nDCG(등급형)   : nDCG@1=1.000 / nDCG@5=0.981 / nDCG@10=0.969 / nDCG@20=0.952
    유사도 margin  : 관련=0.133 − 무관=0.120 = +0.012  (관련 983장 / 무관 217장)
  Qwen파인튜닝      
    Recall(등급≥2): R@1=1.000 / R@5=1.000 / R@10=1.000 / R@20=1.000
    nDCG(등급형)   : nDCG@1=1.000 / nDCG@5=0.984 / nDCG@10=0.977 / nDCG@20=0.970
    유사도 margin  : 관련=0.679 − 무관=0.582 = +0.097  (관련 1071장 / 무관 129장)
  네이버파인튜닝       
    Recall(등급≥2): R@1=0.617 / R@5=0.917 / R@10=0.983 / R@20=0.983
    nDCG(등급형)   : nDCG@1=1.000 / nDCG@5=0.886 / nDCG@10=0.844 / nDCG@20=0.836
    유사도 margin  : 관련=0.669 − 무관=0.625 = +0.043  (관련 598장 / 무관 602장)

유의성 검정 + 오차감소 — 각 모델 vs [원본]
검색어 60개 · Wilcoxon signed-rank (쿼리별 nDCG, paired)

[Qwen파인튜닝] vs [원본]
  nDCG@1 : 평균차=+0.0000 | 승-무-패=0-60-0 | p=  N/A | 오차감소 N/A(천장) → 모든 쿼리 동점 → 검정 불가
  nDCG@5 : 평균차=+0.0030 | 승-무-패=13-35-12 | p=0.904 | 오차감소 +16% → 유의 X · 유효표본 25개
  nDCG@10: 평균차